In [79]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [80]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV, Lasso, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error as MSE
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score
from sklearn.metrics import root_mean_squared_error as RMSE

In [81]:
#lettura dati
df = pd.read_csv('Melbourne_housing.csv')
df.head(6)

C:\Users\m-rog\AppData\Local\Temp\ipykernel_16940\2770047912.py:2: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('Melbourne_housing.csv')


,Suburb,Address,Rooms,Type,Method,SellerG,Date,Distance,Postcode,Bedroom,...,Landsize,BuildingArea,YearBuilt,CouncilArea,Latitude,Longtitude,Regionname,Propertycount,ParkingArea,Price
0,Abbotsford,68 Studley St,2,h,SS,Jellis,3/9/2016,2.5,3067.0,2.0,...,126.0,inf,NaN,Yarra City Council,-37.8014,144.9958,Northern Metropolitan,4019.0,Carport,NaN
1,Airport West,154 Halsey Rd,3,t,PI,Nelson,3/9/2016,13.5,3042.0,3.0,...,303.0,225,2016.0,Moonee Valley City Council,-37.7180,144.8780,Western Metropolitan,3464.0,Detached Garage,840000.0
2,Albert Park,105 Kerferd Rd,2,h,S,hockingstuart,3/9/2016,3.3,3206.0,2.0,...,120.0,82,1900.0,Port Phillip City Council,-37.8459,144.9555,Southern Metropolitan,3280.0,Attached Garage,1275000.0
3,Albert Park,85 Richardson St,2,h,S,Thomson,3/9/2016,3.3,3206.0,2.0,...,159.0,inf,NaN,Port Phillip City Council,-37.8450,144.9538,Southern Metropolitan,3280.0,Indoor,1455000.0
4,Alphington,30 Austin St,3,h,SN,McGrath,3/9/2016,6.4,3078.0,3.0,...,174.0,122,2003.0,Darebin City Council,-37.7818,145.0198,Northern Metropolitan,2211.0,Parkade,NaN
5,Alphington,6 Smith St,4,h,S,Brace,3/9/2016,6.4,3078.0,3.0,...,853.0,263,1930.0,Darebin City Council,-37.7707,145.0318,Northern Metropolitan,2211.0,Underground,2000000.0


In [84]:
df = df.rename(columns={'Longtitude': 'Longitude'})

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
pd.crosstab(index=df['Suburb'], columns='count').reset_index()

In [82]:
null_fraction = (
    df.isnull()
            .mean()
            .rename("fraction_nulls")
            .to_frame()
)

print(null_fraction)

               fraction_nulls
Suburb               0.000000
Address              0.000000
Rooms                0.000000
Type                 0.000000
Method               0.000000
SellerG              0.000000
Date                 0.000000
Distance             0.000029
Postcode             0.000029
Bedroom              0.235735
Bathroom             0.235993
Car                  0.250394
Landsize             0.338813
BuildingArea         0.605244
YearBuilt            0.553863
CouncilArea          0.000086
Latitude             0.228821
Longtitude           0.228821
Regionname           0.000000
Propertycount        0.000086
ParkingArea          0.000000
Price                0.218321


In [ ]:
df.nunique()

In [85]:
columns_to_use = ['Suburb', 'Rooms', 'Type', 'Method', 'SellerG', 'Regionname', 'Propertycount', 'Distance', 'CouncilArea', 'Bedroom', 'Bathroom', 'Car', 'Landsize', 'Price','Longitude', 'Latitude' ]
df_new = df[columns_to_use]
#df_new

In [86]:
df_new = df_new[df_new["Price"].notna()]

In [87]:
null_fraction = (
    df_new.isnull()
            .mean()
            .rename("fraction_nulls")
            .to_frame()
)

print(null_fraction)

               fraction_nulls
Suburb               0.000000
Rooms                0.000000
Type                 0.000000
Method               0.000000
SellerG              0.000000
Regionname           0.000000
Propertycount        0.000110
Distance             0.000037
CouncilArea          0.000110
Bedroom              0.236393
Bathroom             0.236613
Car                  0.250450
Landsize             0.340037
Price                0.000000
Longitude            0.229530
Latitude             0.229530


In [88]:
col_few_nulls = null_fraction[null_fraction["fraction_nulls"] < 0.1].index.tolist()
df_new.dropna(subset=col_few_nulls, inplace=True)

In [89]:
bed_to_room_ratio = (df["Bedroom"]/df["Rooms"]).mean()
bath_to_room_ratio = (df["Bathroom"]/df["Rooms"]).mean()
df_new['Bedroom']  = df_new['Bedroom'].fillna(df['Rooms'] * bed_to_room_ratio )
df_new['Bathroom'] = df_new['Bathroom'].fillna(df['Rooms'] * bath_to_room_ratio)
df_new['Car']      = df_new['Car'].fillna(0)
df_new['Landsize'] = df_new['Landsize'].fillna(df['Landsize'].median())

In [ ]:
(df["Bedroom"]/df["Rooms"]).mean()

In [ ]:
df_new.columns

In [ ]:
df_new['Longitude'] = df_new['Longitude'].fillna(df['Longitude'].median())
df_new['Latitude'] = df_new['Latitude'].fillna(df['Latitude'].median())

In [90]:
# Longitude
df_new['Longitude'] = df_new['Longitude'].fillna(
    df_new.groupby('Suburb')['Longitude'].transform('median')
)
df_new['Longitude'] = df_new['Longitude'].fillna(df_new['Longitude'].median())

# Latitude
df_new['Latitude'] = df_new['Latitude'].fillna(
    df_new.groupby('Suburb')['Latitude'].transform('median')
)
df_new['Latitude'] = df_new['Latitude'].fillna(df_new['Latitude'].median())

In [ ]:
df_new['Longitude']

In [91]:
null_fraction = (
    df_new.isnull()
            .mean()
            .rename("fraction_nulls")
            .to_frame()
)

print(null_fraction)

               fraction_nulls
Suburb                    0.0
Rooms                     0.0
Type                      0.0
Method                    0.0
SellerG                   0.0
Regionname                0.0
Propertycount             0.0
Distance                  0.0
CouncilArea               0.0
Bedroom                   0.0
Bathroom                  0.0
Car                       0.0
Landsize                  0.0
Price                     0.0
Longitude                 0.0
Latitude                  0.0


In [ ]:
df_new.info()

In [92]:
numerical_cols = df_new.select_dtypes(include=["float64","int64"]).columns

scaler = StandardScaler()

df_new[numerical_cols] = scaler.fit_transform(df_new[numerical_cols])

In [ ]:
df_new

In [93]:
# now we are good to go with out cleaned data. Now we are going to make dummy variables for our whole dataset.
df_new = pd.get_dummies(df_new, drop_first=True) # it is a short cut to avoid dummy variable trap it is just dropping the main column whose dummies we have produced. 
df_new

,Rooms,Propertycount,Distance,Bedroom,Bathroom,Car,Landsize,Price,Longitude,Latitude,...,CouncilArea_Moorabool Shire Council,CouncilArea_Moreland City Council,CouncilArea_Nillumbik Shire Council,CouncilArea_Port Phillip City Council,CouncilArea_Stonnington City Council,CouncilArea_Whitehorse City Council,CouncilArea_Whittlesea City Council,CouncilArea_Wyndham City Council,CouncilArea_Yarra City Council,CouncilArea_Yarra Ranges Shire Council
1,0.007996,-0.913292,0.326981,0.021579,0.651325,-0.251407,-0.087088,-0.327695,-1.013831,0.987283,...,False,False,False,False,False,False,False,False,False,False
2,-1.039352,-0.954251,-1.175792,-1.018225,-0.857440,-1.130765,-0.147037,0.350424,-0.357698,-0.418652,...,False,False,False,True,False,False,False,False,False,False
3,-1.039352,-0.954251,-1.175792,-1.018225,-0.857440,-1.130765,-0.134261,0.631025,-0.372091,-0.408759,...,False,False,False,True,False,False,False,False,False,False
5,1.055344,-1.192214,-0.719067,0.021579,0.651325,2.386669,0.093086,1.480622,0.288275,0.407981,...,False,False,False,False,False,False,False,False,False,False
6,0.007996,-1.192214,-0.719067,0.021579,0.651325,0.627952,-0.118209,0.093206,0.294201,0.246392,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34851,0.007996,0.943663,1.682422,0.021579,-0.857440,4.145385,0.000051,-0.658960,1.562103,-1.771823,...,False,False,False,False,False,False,False,False,False,False
34852,0.007996,3.134969,0.105985,0.021579,-0.857440,-0.251407,-0.015674,-0.896691,0.216481,1.210540,...,False,False,False,False,False,False,False,False,False,False
34853,1.055344,-0.385945,1.373028,1.061382,0.651325,0.627952,-0.015674,-0.715860,-0.575619,1.881519,...,False,False,False,False,False,False,False,False,False,False
34855,0.007996,-0.781956,1.608757,0.021579,0.651325,-0.251407,-0.008139,-0.420449,1.273151,-1.865368,...,False,False,False,False,False,False,False,False,False,False


In [94]:
y = df_new.Price
X = df_new.drop('Price', axis='columns')

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size = 0.8, random_state = 42)


In [95]:
regression = LinearRegression()
regression.fit(X_train, y_train)
pred = regression.predict(X_test)

errore = MSE(y_test, pred)
errore_sqr = RMSE(y_test, pred)
r2 = r2_score(y_test, pred)

print("R2 Score:", r2)
print("Mean Squared Error:", errore)
print("Root Mean Squared Error:", errore_sqr)
print()

ridge_coefs = pd.DataFrame({
    "feature": X_train.columns.values,
    "coef_ridge": regression.coef_
}).sort_values("coef_ridge", key=abs, ascending=False)

print("\nTop 10 coefficienti Ridge (per valore assoluto):")
print(ridge_coefs.head(10))

R2 Score: 0.6557991761982014
Mean Squared Error: 0.3274362547740769
Root Mean Squared Error: 0.5722204599401152


Top 10 coefficienti Ridge (per valore assoluto):
                feature  coef_ridge
439      SellerG_Darras    3.079924
460         SellerG_For    2.562753
487      SellerG_Hooper    2.237850
640   SellerG_Sotheby's    2.158287
476        SellerG_Hall    2.090576
168  Suburb_Healesville    1.487232
566    SellerG_Nicholas    1.374053
674       SellerG_Weast    1.340578
392        SellerG_Blue    1.287351
664     SellerG_VICProp    1.246231


In [96]:
lasso_model = Lasso(alpha = 0.01)
lasso_model.fit(X_train, y_train)

y_pred_lasso = lasso_model.predict(X_test)
lasso_r2 = r2_score(y_test, y_pred_lasso)
lasso_mse = MSE(y_test, y_pred_lasso)
lasso_rmse = np.sqrt(lasso_mse)

print("\n===== RISULTATI LASSO =====")

print("R2 test:", round(lasso_r2, 4))
print("MSE test:", round(lasso_mse, 6))
print("RMSE test:", round(lasso_rmse, 4))


===== RISULTATI LASSO =====
R2 test: 0.5843
MSE test: 0.395437
RMSE test: 0.6288


In [97]:
ridge_model = Ridge(alpha = 20)
ridge_model.fit(X_train, y_train)

pred = ridge_model.predict(X_test)
ridge_r2 = r2_score(y_test, pred)
ridge_mse = MSE(y_test, pred)
ridge_rmse = np.sqrt(ridge_mse)

print("\n===== RISULTATI RIDGE =====")

print("R2 test:", round(ridge_r2, 4))
print("MSE test:", round(ridge_mse, 6))
print("RMSE test:", round(ridge_rmse, 4))


===== RISULTATI RIDGE =====
R2 test: 0.6632
MSE test: 0.32038
RMSE test: 0.566


In [98]:
# ==============================
# Ridge Regression
# ==============================

alphas_ridge = np.linspace(0.1, 30, 30)
ridge = RidgeCV(alphas=alphas_ridge, cv=5)
ridge.fit(X_train, y_train)

y_pred_ridge = ridge.predict(X_test)

ridge_r2 = r2_score(y_test, y_pred_ridge)
ridge_mse = MSE(y_test, y_pred_ridge)
ridge_rmse = np.sqrt(ridge_mse)

print("\n===== RISULTATI RIDGE =====")
print("Alpha scelto:", ridge.alpha_)
print("R2 test:", round(ridge_r2, 4))
print("MSE test:", round(ridge_mse, 6))
print("RMSE test:", round(ridge_rmse, 4))

ridge_coefs = pd.DataFrame({
    "feature": X_train.columns.values,
    "coef_ridge": ridge.coef_
}).sort_values("coef_ridge", key=abs, ascending=False)

print("\nTop 10 coefficienti Ridge (per valore assoluto):")
print(ridge_coefs.head(10))



===== RISULTATI RIDGE =====
Alpha scelto: 6.2862068965517235
R2 test: 0.664
MSE test: 0.319633
RMSE test: 0.5654

Top 10 coefficienti Ridge (per valore assoluto):
                  feature  coef_ridge
640     SellerG_Sotheby's    1.236332
476          SellerG_Hall    1.047731
222    Suburb_Middle Park    0.842584
70      Suburb_Canterbury    0.759049
181   Suburb_Ivanhoe East    0.745954
353                Type_u   -0.726087
508           SellerG_Kay    0.673536
299        Suburb_Sunbury    0.641361
123      Suburb_Eaglemont    0.604550
360  SellerG_Abercromby's    0.593097


In [ ]:
# ==============================
# Lasso Regression
# ==============================

alphas_lasso = np.logspace(-10, 100, 10)
lasso = LassoCV(alphas=alphas_ridge, cv=5)
lasso.fit(X_train, y_train)

y_pred_ridge = ridge.predict(X_test)

lasso_r2 = r2_score(y_test, y_pred_ridge)
lasso_mse = MSE(y_test, y_pred_ridge)
lasso_rmse = np.sqrt(ridge_mse)

print("\n===== RISULTATI RIDGE =====")
print("Alpha scelto:", ridge.alpha_)
print("R2 test:", round(lasso_r2, 4))
print("MSE test:", round(lasso_mse, 6))
print("RMSE test:", round(lasso_rmse, 4))

ridge_coefs = pd.DataFrame({
    "feature": X_train.columns.values,
    "coef_ridge": ridge.coef_
}).sort_values("coef_ridge", key=abs, ascending=False)

print("\nTop 10 coefficienti Ridge (per valore assoluto):")
print(ridge_coefs.head(10))